In [2]:
# ==========================================================
# IMPORTS
# ==========================================================
import os
import sys
import warnings
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from dotenv import load_dotenv

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_mistralai import ChatMistralAI

from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
)

# Ignore certains warnings de compatibilité
warnings.filterwarnings("ignore", category=DeprecationWarning)


# ==========================================================
# CONFIGURATION GÉNÉRALE
# ==========================================================
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

load_dotenv()

TOKEN = os.getenv("API_KEY", "")
TOKEN_MISTRAL = os.getenv("MISTRAL_API_KEY", "")
TOKEN_OPENAGENDA = os.getenv("OPENAGENDA_API_KEY", "")

API_URL_HEALTH = "http://127.0.0.1:8000/health"
API_URL_REBUILD = "http://127.0.0.1:8000/rebuild"
API_URL_ASK = "http://127.0.0.1:8000/ask"
API_URL_ASK_DEBUG = "http://127.0.0.1:8000/ask/debug"

# Vérifications minimales
if not TOKEN:
    raise ValueError("API_KEY manquante dans le fichier .env")

if not TOKEN_MISTRAL:
    raise ValueError("MISTRAL_API_KEY manquante dans le fichier .env")


# ==========================================================
# FONCTIONS UTILITAIRES API
# ==========================================================
def normalize_text(text: str) -> str:
    """
    Normalise une chaîne pour comparaison souple.

    Parameters
    ----------
    text : str
        Texte à normaliser.

    Returns
    -------
    str
        Texte nettoyé.
    """
    if not isinstance(text, str):
        return ""
    return " ".join(text.strip().lower().split())


def ensure_api_ready(zone: str = "Montpellier", scope: str = "city") -> None:
    """
    Vérifie si l'API a déjà un index chargé.
    Si ce n'est pas le cas, déclenche une reconstruction.

    Parameters
    ----------
    zone : str, default="Montpellier"
        Zone à charger si un rebuild est nécessaire.
    scope : str, default="city"
        Scope utilisé pour le rebuild.

    Raises
    ------
    RuntimeError
        Si l'API reste indisponible après tentative de rebuild.
    """
    try:
        resp = requests.get(API_URL_HEALTH, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        if data.get("index_loaded", False):
            print("API prête : index déjà chargé.")
            return

        print("Index non chargé. Lancement de /rebuild ...")

        rebuild_resp = requests.post(
            API_URL_REBUILD,
            json={"zone": zone, "scope": scope},
            headers={"x-api-key": TOKEN},
            timeout=300,
        )
        rebuild_resp.raise_for_status()

        print("Rebuild terminé :", rebuild_resp.json())

        final_resp = requests.get(API_URL_HEALTH, timeout=30)
        final_resp.raise_for_status()
        final_data = final_resp.json()

        if not final_data.get("index_loaded", False):
            raise RuntimeError("L'index n'est toujours pas chargé après /rebuild.")

        print("API prête après rebuild.")

    except requests.RequestException as exc:
        raise RuntimeError("Impossible de préparer l'API pour l'évaluation.") from exc


def call_ask_api(question: str) -> dict[str, Any]:
    """
    Appelle l'endpoint /ask de l'API RAG.

    Parameters
    ----------
    question : str
        Question utilisateur.

    Returns
    -------
    dict[str, Any]
        Réponse JSON brute de l'API.
    """
    resp = requests.post(
        API_URL_ASK,
        json={"question": question},
        headers={"x-api-key": TOKEN},
        timeout=60,
    )

    print("STATUS /ask :", resp.status_code)

    data = resp.json()
    print("ANSWER /ask :")
    print(data.get("answer", ""))

    return data


def run_rag_for_eval(question: str) -> dict[str, Any]:
    """
    Appelle l'endpoint /ask/debug afin de récupérer :
    - la réponse générée
    - les contextes récupérés
    - les documents utilisés
    - le nombre de documents
    - une éventuelle erreur

    Important :
    `/ask/debug` renvoie `retrieved_contexts` sous forme de liste de contextes,
    avec un contexte textuel par document.

    Parameters
    ----------
    question : str
        Question à poser à l'API.

    Returns
    -------
    dict[str, Any]
        Résultat normalisé exploitable pour l'évaluation.
    """
    try:
        resp = requests.post(
            API_URL_ASK_DEBUG,
            json={"question": question},
            headers={"x-api-key": TOKEN},
            timeout=180,
        )

        print(f"QUESTION : {question}")
        print("STATUS /ask/debug :", resp.status_code)

        data = resp.json()

        if not resp.ok:
            return {
                "question": question,
                "answer": None,
                "contexts": None,
                "documents": None,
                "n_docs": 0,
                "error": data.get("detail", f"HTTP {resp.status_code}"),
            }

        return {
            "question": question,
            "answer": data.get("answer", ""),
            "contexts": data.get("retrieved_contexts", []),
            "documents": data.get("documents", []),
            "n_docs": data.get("n_docs", 0),
            "error": None,
        }

    except requests.exceptions.RequestException as exc:
        return {
            "question": question,
            "answer": None,
            "contexts": None,
            "documents": None,
            "n_docs": 0,
            "error": str(exc),
        }


# ==========================================================
# PRÉPARATION DE L'API
# ==========================================================
ensure_api_ready(zone="Montpellier", scope="city")

# ==========================================================
# JEU D'ÉVALUATION
# Version RAGAS-friendly
# ==========================================================
eval_questions_positive = [
    {
        "question": "Y a-t-il des expositions à Montpellier ?",
        "ground_truth": (
            "Liste d'événements pour Y a-t-il des expositions à Montpellier ?: \n"
            "- L'Ecole des Beaux-Arts de Montpellier : une histoire singulière "
            "(date : 2026/01/31 au 2026/03/08, lieu : MO.CO. Hôtel des collections).\n"
            "- Exposition et atelier portes ouvertes Miss'terre JAA 8ème édition "
            "(date : 2025/10/11 au 2025/10/12, lieu : Association Miss'terre).\n"
            "- VERNISSAGE EXPOSITION | « La graaande traversée » "
            "(date : 2026/02/26, lieu : CAUE de l'Hérault)."
        ),
        "group": "positive",
    },
    {
        "question": "Quels événements ont lieu à Montpellier en septembre 2025 ?",
        "ground_truth": (
            "Liste d'événements pour Quels événements ont lieu à Montpellier en septembre 2025 ?:\n"
            "- Concert : ZOHUD "
            "(date : 2025/09/21, lieu : Domaine D’O).\n"
            "- Contes : JIHAD DARWICHE "
            "(date : 2025/09/20 au 2025/09/21, lieu : Domaine d'O)."
            "- Concert : Le DUO SABIL & VINCENT SEGAL "
            "(date : 2025/09/13, lieu : Domaine d'O)"
        ),
        "group": "positive",
    },
    {
        "question": "Quels événements ont lieu le 20 septembre 2025 à Montpellier ?",
        "ground_truth": (
            "Liste d'événements pour Quels événements ont lieu le 20 septembre 2025 à Montpellier ?:\n"
            "- Contes : JIHAD DARWICHE "
            "(date : 2025/09/20 au 2025/09/21, lieu : Domaine d'O)."
        ),
        "group": "positive",
    },
    {
        "question": "Quels événements ont lieu le week-end du 20 au 21 septembre 2025 à Montpellier ?",
        "ground_truth": (
            "Liste d'événements pour Quels événements ont lieu le week-end du 20 au 21 septembre 2025 à Montpellier ?:\n"
            "- Concert : ZOHUD "
            "(date : 2025/09/21, lieu : Domaine D’O).\n"
            "- Contes : JIHAD DARWICHE "
            "(date : 2025/09/20 au 2025/09/21, lieu : Domaine d'O).\n"
            "- Chez le Poët Jeannot "
            "(date : 2025/09/21, lieu : Chez Jeannot).\n"
        ),
        "group": "positive"
    }
]

eval_questions_ambiguous = [
    {
        "question": "Quels événements culturels sont proposés à Montpellier ?",
        "ground_truth": (
            "Liste d'événements pour Quels événements culturels sont proposés à Montpellier ?:\n"
            "- 12° Vinyl Pop-Up de Montpellier "
            "(date : 2026/03/29, lieu : La Halle tropisme).\n"
            "- Week-end \"Éclosion\" - 4 pratiques transformatrices ! "
            "(date : 2025/12/06 au 2025/12/07, lieu : Centre de l'être).\n"
            "- Les stages vacances de programmation d'hiver à Montpellier sont là ! "
            "(date : 2025/12/22 au 2025/12/24, lieu : La Halle tropisme)."
        ),
        "group": "ambiguous",
    },
    {
        "question": "Que faire à Montpellier le week-end du 28 et 29 mars 2026?",
        "ground_truth": (
            "Liste d'événements pour Que faire à Montpellier le week-end du 28 et 29 mars 2026?:\n"
            "- 12° Vinyl Pop-Up de Montpellier "
            "(date : 2026/03/29, lieu : La Halle tropisme)."
        ),
        "group": "ambiguous",
    },
    {
        "question": "Quels événements gratuits ont lieu à Montpellier ?",
        "ground_truth": (
            "Liste d'événements pour Quels événements gratuits ont lieu à Montpellier ?:\n"
            "- GRAND DÉSTOCKAGE - Braderie à Montpellier "
            "(date : 2025/12/06, lieu : Marché gare).\n"
            "- Journée Portes Ouvertes de la Faculté d’Éducation de l’Université de Montpellier "
            "(date : 2026/02/14, lieu : Faculté d’Éducation).\n"
            "- Je participe aux ateliers de réparations collectives "
            "(date : 2025/10/18, lieu : REPAIR CAFÉ MONTPELLIER)."
        ),
        "group": "ambiguous",
    },
    {
        "question": "Y a-t-il des concerts de rock à Montpellier ?",
        "ground_truth": (
            "Liste d'événements pour Y a-t-il des concerts de rock à Montpellier ?:\n"
            "- EX TENEBRIS LUX - BRANT BJORK TRIO + BRONCA "
            "(date : 2025/11/05, lieu : L'Antirouille).\n"
            "- Concert : ZOHUD "
            "(date : 2025/09/21, lieu : Domaine d'O).\n"
            "- Mr Jack en concert au Gazette Café "
            "(date : 2026/03/14, lieu : Gazette Café)."
        ),
        "group": "ambiguous",
    }
]

eval_questions_negative = [
    {
        "question": "Quels événements ont lieu à Paris ?",
        "ground_truth": (
            "Je n'ai trouvé aucun événement correspondant dans les données disponibles."
        ),
        "group": "negative",
    },
]

eval_questions = (
    eval_questions_positive
    + eval_questions_ambiguous
    + eval_questions_negative
)

print("Nombre total de questions d'évaluation :", len(eval_questions))

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# ==========================================================
# COLLECTE DES RÉSULTATS DE L'API
# ==========================================================
rows = []

for item in eval_questions:
    result = run_rag_for_eval(item["question"])

    rows.append(
        {
            "question": item["question"],
            "answer": result["answer"],
            "contexts": result["contexts"],
            "documents": result["documents"],
            "n_docs": result["n_docs"],
            "ground_truth": item["ground_truth"],
            "group": item["group"],
            "error": result["error"],
        }
    )

rows_df = pd.DataFrame(rows)

print("\nAperçu des résultats bruts :")
display(rows_df)

# Vérification rapide des erreurs API
error_rows = rows_df[rows_df["error"].notna()].copy()
print("Nombre d'erreurs :", len(error_rows))

if not error_rows.empty:
    display(error_rows[["question", "error"]])


# ==========================================================
# CONSTRUCTION DU DATASET RAGAS
# ==========================================================
ragas_rows = [
    {
        "user_input": row["question"],
        "response": row["answer"],
        "retrieved_contexts": row["contexts"],
        "reference": row["ground_truth"],
        "group": row["group"],
    }
    for row in rows
    if row["answer"] is not None and row["contexts"] is not None
]

print("Nombre de lignes exploitables pour RAGAS :", len(ragas_rows))

if not ragas_rows:
    raise ValueError("Aucune ligne exploitable pour RAGAS.")

print("\nExemple de ligne RAGAS :")
print(ragas_rows[0])


# ==========================================================
# CONFIGURATION DES ÉVALUATEURS
# ==========================================================
llm = ChatMistralAI(
    model="mistral-small-latest",
    api_key=TOKEN_MISTRAL.strip(),
    temperature=0,
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)


# ==========================================================
# SPLIT DU DATASET PAR GROUPE
# ==========================================================
rows_positive = [row for row in ragas_rows if row["group"] == "positive"]
rows_ambiguous = [row for row in ragas_rows if row["group"] == "ambiguous"]
rows_negative = [row for row in ragas_rows if row["group"] == "negative"]

dataset_positive = EvaluationDataset.from_list(rows_positive)
dataset_ambiguous = EvaluationDataset.from_list(rows_ambiguous)

print("\nNombre de questions positives :", len(rows_positive))
print("Nombre de questions ambiguës :", len(rows_ambiguous))
print("Nombre de questions négatives :", len(rows_negative))


# ==========================================================
# ÉVALUATION RAGAS - QUESTIONS POSITIVES
# ==========================================================
result_positive = evaluate(
    dataset=dataset_positive,
    metrics=[
        Faithfulness(),
        AnswerRelevancy(strictness=1),
        ContextPrecision(),
        ContextRecall(),
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=False,
)

df_positive = result_positive.to_pandas().reset_index(drop=True)
df_positive["group_eval"] = "positive"
df_positive["question"] = [row["user_input"] for row in rows_positive]

print("\nRésultats RAGAS - Positives :")
display(df_positive)


# ==========================================================
# ÉVALUATION RAGAS - QUESTIONS AMBIGUËS
# ==========================================================
result_ambiguous = evaluate(
    dataset=dataset_ambiguous,
    metrics=[
        Faithfulness(),
        AnswerRelevancy(strictness=1),
        ContextPrecision(),
        ContextRecall(),
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=False,
)

df_ambiguous = result_ambiguous.to_pandas().reset_index(drop=True)
df_ambiguous["group_eval"] = "ambiguous"
df_ambiguous["question"] = [row["user_input"] for row in rows_ambiguous]

print("\nRésultats RAGAS - Ambiguës :")
display(df_ambiguous)


# ==========================================================
# ÉVALUATION MÉTIER - QUESTIONS NÉGATIVES
# ==========================================================
df_negative_business = rows_df[rows_df["group"] == "negative"].copy()

df_negative_business["negative_exact_match"] = df_negative_business.apply(
    lambda row: int(
        normalize_text(row["answer"]) == normalize_text(row["ground_truth"])
    ),
    axis=1,
)

negative_score = df_negative_business["negative_exact_match"].mean()

print("\nScore métier négatif :", negative_score)
display(
    df_negative_business[
        ["question", "answer", "ground_truth", "negative_exact_match"]
    ]
)


# ==========================================================
# TABLEAUX RÉCAPITULATIFS
# ==========================================================
positive_mean = df_positive[
    ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
].mean()

ambiguous_mean = df_ambiguous[
    ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
].mean()

results_summary = {
    "positive_mean": positive_mean,
    "ambiguous_mean": ambiguous_mean,
    "negative_exact_match_mean": negative_score,
}

print("\nRésumé des scores :")
for key, value in results_summary.items():
    print(f"\n{key}")
    print(value)


# ==========================================================
# DATAFRAME DE SYNTHÈSE
# ==========================================================
summary_df = pd.DataFrame({
    "group": ["positive", "ambiguous", "negative"],
    "faithfulness": [
        df_positive["faithfulness"].mean(),
        df_ambiguous["faithfulness"].mean(),
        None,
    ],
    "answer_relevancy": [
        df_positive["answer_relevancy"].mean(),
        df_ambiguous["answer_relevancy"].mean(),
        None,
    ],
    "context_precision": [
        df_positive["context_precision"].mean(),
        df_ambiguous["context_precision"].mean(),
        None,
    ],
    "context_recall": [
        df_positive["context_recall"].mean(),
        df_ambiguous["context_recall"].mean(),
        None,
    ],
    "negative_exact_match": [
        None,
        None,
        negative_score,
    ],
})

print("\nTableau de synthèse final :")
display(summary_df)


# ==========================================================
# BONUS : TABLEAUX DÉTAILLÉS AVEC QUESTION + SCORE
# ==========================================================
positive_detailed = pd.DataFrame({
    "question": [row["user_input"] for row in rows_positive],
    "faithfulness": df_positive["faithfulness"],
    "answer_relevancy": df_positive["answer_relevancy"],
    "context_precision": df_positive["context_precision"],
    "context_recall": df_positive["context_recall"],
})

ambiguous_detailed = pd.DataFrame({
    "question": [row["user_input"] for row in rows_ambiguous],
    "faithfulness": df_ambiguous["faithfulness"],
    "answer_relevancy": df_ambiguous["answer_relevancy"],
    "context_precision": df_ambiguous["context_precision"],
    "context_recall": df_ambiguous["context_recall"],
})

print("\nDétail des scores - Positives :")
display(positive_detailed)

print("\nDétail des scores - Ambiguës :")
display(ambiguous_detailed)


API prête : index déjà chargé.
Nombre total de questions d'évaluation : 9
QUESTION : Y a-t-il des expositions à Montpellier ?
STATUS /ask/debug : 200
QUESTION : Quels événements ont lieu à Montpellier en septembre 2025 ?
STATUS /ask/debug : 200
QUESTION : Quels événements ont lieu le 20 septembre 2025 à Montpellier ?
STATUS /ask/debug : 200
QUESTION : Quels événements ont lieu le week-end du 20 au 21 septembre 2025 à Montpellier ?
STATUS /ask/debug : 200
QUESTION : Quels événements culturels sont proposés à Montpellier ?
STATUS /ask/debug : 200
QUESTION : Que faire à Montpellier le week-end du 28 et 29 mars 2026?
STATUS /ask/debug : 200
QUESTION : Quels événements gratuits ont lieu à Montpellier ?
STATUS /ask/debug : 200
QUESTION : Y a-t-il des concerts de rock à Montpellier ?
STATUS /ask/debug : 200
QUESTION : Quels événements ont lieu à Paris ?
STATUS /ask/debug : 200

Aperçu des résultats bruts :


,question,answer,contexts,documents,n_docs,ground_truth,group,error
0,Y a-t-il des expositions à Montpellier ?,"Liste d'événements pour Y a-t-il des expositions à Montpellier ? :\n- L'Ecole des Beaux-Arts de Montpellier : une histoire singulière (date : 2026/01/31 au 2026/03/08, lieu : MO.CO. Hôtel des collections)\n- Exposition et atelier portes ouvertes Miss'terre JAA 8ème édition (date : 2025/10/11 au 2025/10/12, lieu : Association Miss'terre)\n- VERNISSAGE EXPOSITION | « La graaande traversée » (date : 2026/02/26, lieu : CAUE de l'Hérault)","[Événement 1\nTitre : L'Ecole des Beaux-Arts de Montpellier : une histoire singulière\nLieu : MO.CO. Hôtel des collections\nVille : Montpellier\nRégion : Occitanie\nDate de début : 2026-01-31\nDate de fin : 2026-03-08\nType : exposition\nGenre musical : \nTarification : inconnu\nDescription : Pour cette 3ème édition de ""SOL ! La biennale du territoire"", le MO.CO. et le musée Fabre nouent un partenariat exceptionnel pour rendre hommage à l'École des beaux-arts de Montpellier.\nURL : https://openagenda.com/air-de-midi-reseau-art-contemporain-en-occitanie/events/lecole-des-beaux-arts-de-montpellier-une-histoire-singuliere, Événement 2\nTitre : Exposition et atelier portes ouvertes Miss'terre JAA 8ème édition\nLieu : Association Miss'terre\nVille : Montpellier\nRégion : Occitanie\nDate de début : 2025-10-11\nDate de fin : 2025-10-12\nType : exposition\nGenre musical : \nTarification : gratuit\nDescription : À l'occasion de la 8ème édition des journées ateliers d'artistes Occitanie, l'atelier MissTerre ouvre ses portes au public, venez découvrir le travail des artistes de votre région !\nURL : https://openagenda.com/contributeurs-petit-agenda/events/exposition-et-atelier-portes-ouvertes-missterre-jaa-8eme-edition, Événement 3\nTitre : VERNISSAGE EXPOSITION | « La graaande traversée »\nLieu : CAUE de l'Hérault\nVille : Montpellier\nRégion : Occitanie\nDate de début : 2026-02-26\nDate de fin : 2026-02-26\nType : exposition\nGenre musical : \nTarification : gratuit\nDescription : La MAOM vous invite le jeudi 26 février à 18h30 à découvrir l’exposition « La graaande traversée », portée par le duo Alice Migeon, artiste-paysagiste et Alice Rochette, designeuse !\nURL : https://openagenda.com/calendrier-annuel-de-l-architecture-mcc/events/vernissage-exposition-or-la-graaande-traversee]","[{'title': 'L'Ecole des Beaux-Arts de Montpellier : une histoire singulière', 'location_name': 'MO.CO. Hôtel des collections', 'city': 'Montpellier', 'region': 'Occitanie', 'first_date': '2026-01-31', 'last_date': '2026-03-08', 'event_type': 'exposition', 'url': 'https://openagenda.com/air-de-midi-reseau-art-contemporain-en-occitanie/events/lecole-des-beaux-arts-de-montpellier-une-histoire-singuliere', 'price_info': 'inconnu', 'is_free': None, 'keywords_title': ['arts', 'beaux', 'ecole', 'histoire', 'montpellier', 'singuliere'], 'score': 22.250661945343015}, {'title': 'Exposition et atelier portes ouvertes Miss'terre JAA 8ème édition', 'location_name': 'Association Miss'terre', 'city': 'Montpellier', 'region': 'Occitanie', 'first_date': '2025-10-11', 'last_date': '2025-10-12', 'event_type': 'exposition', 'url': 'https://openagenda.com/contributeurs-petit-agenda/events/exposition-et-atelier-portes-ouvertes-missterre-jaa-8eme-edition', 'price_info': 'gratuit', 'is_free': True, 'keywords_title': ['8eme', 'atelier', 'edition', 'exposition', 'jaa', 'miss', 'ouvertes', 'portes', 'terre'], 'score': 21.487860012054444}, {'title': 'VERNISSAGE EXPOSITION | « La graaande traversée »', 'location_name': 'CAUE de l'Hérault', 'city': 'Montpellier', 'region': 'Occitanie', 'first_date': '2026-02-26', 'last_date': '2026-02-26', 'event_type': 'exposition', 'url': 'https://openagenda.com/calendrier-annuel-de-l-architecture-mcc/events/vernissage-exposition-or-la-graaande-traversee', 'price_info': 'gratuit', 'is_free': True, 'keywords_title': ['exposition', 'graaande', 'traversee', 'vernissage'], 'score': 21.41482856273651}]",3,"List

Nombre d'erreurs : 0
Nombre de lignes exploitables pour RAGAS : 9

Exemple de ligne RAGAS :
{'user_input': 'Y a-t-il des expositions à Montpellier ?', 'response': "Liste d'événements pour Y a-t-il des expositions à Montpellier ? :\n- L'Ecole des Beaux-Arts de Montpellier : une histoire singulière (date : 2026/01/31 au 2026/03/08, lieu : MO.CO. Hôtel des collections)\n- Exposition et atelier portes ouvertes Miss'terre JAA 8ème édition (date : 2025/10/11 au 2025/10/12, lieu : Association Miss'terre)\n- VERNISSAGE EXPOSITION | « La graaande traversée » (date : 2026/02/26, lieu : CAUE de l'Hérault)", 'retrieved_contexts': ['Événement 1\nTitre : L\'Ecole des Beaux-Arts de Montpellier : une histoire singulière\nLieu : MO.CO. Hôtel des collections\nVille : Montpellier\nRégion : Occitanie\nDate de début : 2026-01-31\nDate de fin : 2026-03-08\nType : exposition\nGenre musical : \nTarification : inconnu\nDescription : Pour cette 3ème édition de "SOL ! La biennale du territoire", le MO.CO. et le 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

,group,faithfulness,answer_relevancy,context_precision,context_recall,negative_exact_match
0,positive,0.972222,1.000000,0.833333,0.6875,NaN
1,ambiguous,1.000000,0.687367,0.645833,0.5000,NaN
2,negative,NaN,NaN,NaN,NaN,1.0



Détail des scores - Positives :


,question,faithfulness,answer_relevancy,context_precision,context_recall
0,Y a-t-il des expositions à Montpellier ?,0.888889,1.0,1.000000,0.75
1,Quels événements ont lieu à Montpellier en septembre 2025 ?,1.000000,1.0,0.333333,0.75
2,Quels événements ont lieu le 20 septembre 2025 à Montpellier ?,1.000000,1.0,1.000000,0.50
3,Quels événements ont lieu le week-end du 20 au 21 septembre 2025 à Montpellier ?,1.000000,1.0,1.000000,0.75



Détail des scores - Ambiguës :


,question,faithfulness,answer_relevancy,context_precision,context_recall
0,Quels événements culturels sont proposés à Montpellier ?,1.0,1.000000,0.000000,0.00
1,Que faire à Montpellier le week-end du 28 et 29 mars 2026?,1.0,0.573809,1.000000,0.50
2,Quels événements gratuits ont lieu à Montpellier ?,1.0,1.000000,0.583333,0.75
3,Y a-t-il des concerts de rock à Montpellier ?,1.0,0.175658,1.000000,0.75
